# Benchmark A - DeBERTa-v3-base
https://doi.org/10.18653/v1/2024.nllp-1.34

In [4]:
%pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.13.0
Uninstalling torch-2.13.0:
  Successfully uninstalled torch-2.13.0
Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


In [5]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 16.1 MB/s eta 0:02:52
   ---------------------------------------- 0.0/2.8 GB 16.6 MB/s eta 0:02:46
   ---------------------------------------- 0.0/2.8 GB 17.2 MB/s eta 0:02:40
   ---------------------------------------- 0.0/2.8 GB 17.2 MB/s eta 0:02:40
   ---------------------------------------- 0.0/2.8 GB 17.4 MB/s eta 0:02:38
   ---------------------------------------- 0.0/2.8 GB 17.5 MB/s eta 0:02:36
   ---------------------------------------- 0.0/2.8 GB 17.6 MB/s eta 0:02:35
   ---------------------------------------- 0.0/2.8 GB 17.6 MB/s eta 0:02:35
   ---------------------------------------- 0.0/2.8 GB 17.7 MB/s eta 0:02:35
    --------------------------------------- 0.0/2.8 GB 17.5 MB/s eta 0:02:35
    --------------------------------------- 0.0/2.8 GB 17.5 MB/s eta 0:02:36
    ---------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
# Import libraries
import os
import json
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch version: 2.11.0+cu128
CUDA version: 12.8
CUDA available: True
GPU: NVIDIA GeForce RTX 3050
VRAM: 6.0 GB


In [3]:
# Load the train, validation, and test datasets from JSON files
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_records = load_json("../data/train.json")
val_records = load_json("../data/val.json")
test_records = load_json("../data/test.json")

print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

Train: 176
Validation: 22
Test: 22


In [5]:
# Define entity types and create label mappings
ENTITY_TYPES = [
    "NAME",
    "COLLEGE_NAME",
    "COMPANY",
    "DEGREE",
    "DESIGNATION",
    "EMAIL",
    "GRADUATION_YEAR",
    "LOCATION",
    "SKILLS",
    "YEARS_OF_EXPERIENCE"
]

LABEL_LIST = ["O"]

for entity in ENTITY_TYPES:
    LABEL_LIST.append(f"B-{entity}")
    LABEL_LIST.append(f"I-{entity}")

label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for label, i in label2id.items()}

print("Number of labels:", len(LABEL_LIST))
print(LABEL_LIST)

Number of labels: 21
['O', 'B-NAME', 'I-NAME', 'B-COLLEGE_NAME', 'I-COLLEGE_NAME', 'B-COMPANY', 'I-COMPANY', 'B-DEGREE', 'I-DEGREE', 'B-DESIGNATION', 'I-DESIGNATION', 'B-EMAIL', 'I-EMAIL', 'B-GRADUATION_YEAR', 'I-GRADUATION_YEAR', 'B-LOCATION', 'I-LOCATION', 'B-SKILLS', 'I-SKILLS', 'B-YEARS_OF_EXPERIENCE', 'I-YEARS_OF_EXPERIENCE']


In [7]:
# Load the tokenizer for the DeBERTa model
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("Tokenizer loaded:", MODEL_NAME)

Tokenizer loaded: microsoft/deberta-v3-base


In [8]:
# Convert character annotations to token BIO labels
MAX_LENGTH = 256
STRIDE = 32

def prepare_ner_dataset(records):
    processed = []

    for record in records:
        text = record["content"]

        # Convert DataTurks annotations into character spans
        entities = []

        for ann in record.get("annotation", []):
            if not ann.get("label"):
                continue

            entity_type = ann["label"][0]

            for point in ann.get("points", []):
                start = point["start"]

                # DataTurks end position is inclusive
                end = point["end"] + 1

                entities.append({
                    "start": start,
                    "end": end,
                    "label": entity_type
                })

        entities = sorted(
            entities,
            key=lambda x: (x["start"], x["end"])
        )

        encoded = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            stride=STRIDE,
            return_overflowing_tokens=True,
            return_offsets_mapping=True
        )

        number_of_chunks = len(encoded["input_ids"])

        for chunk_index in range(number_of_chunks):

            offsets = encoded["offset_mapping"][chunk_index]

            token_labels = []
            previous_entity = None

            for start_offset, end_offset in offsets:

                # Special tokens such as [CLS] and [SEP]
                if start_offset == 0 and end_offset == 0:
                    token_labels.append(-100)
                    previous_entity = None
                    continue

                matched_entity = None

                for entity in entities:
                    # Check whether token overlaps an annotated entity
                    if (
                        end_offset > entity["start"]
                        and start_offset < entity["end"]
                    ):
                        matched_entity = entity
                        break

                if matched_entity is None:
                    token_labels.append(label2id["O"])
                    previous_entity = None

                else:
                    entity_type = matched_entity["label"]

                    entity_identity = (
                        matched_entity["start"],
                        matched_entity["end"],
                        entity_type
                    )

                    if previous_entity == entity_identity:
                        bio_label = f"I-{entity_type}"
                    else:
                        bio_label = f"B-{entity_type}"

                    token_labels.append(label2id[bio_label])
                    previous_entity = entity_identity

            item = {
                "input_ids": encoded["input_ids"][chunk_index],
                "attention_mask": encoded["attention_mask"][chunk_index],
                "labels": token_labels
            }

            processed.append(item)

    return Dataset.from_list(processed)

In [9]:
# Process train / validation / test
train_dataset = prepare_ner_dataset(train_records)
val_dataset = prepare_ner_dataset(val_records)
test_dataset = prepare_ner_dataset(test_records)

print("Training chunks:", len(train_dataset))
print("Validation chunks:", len(val_dataset))
print("Testing chunks:", len(test_dataset))

Training chunks: 327
Validation chunks: 39
Testing chunks: 41


In [10]:
from collections import Counter

label_counter = Counter()

for row in train_dataset:
    for label_id in row["labels"]:
        if label_id != -100:
            label_counter[id2label[label_id]] += 1

print("Training BIO Label Distribution:")

for label, count in label_counter.most_common():
    print(f"{label:<30} {count}")

Training BIO Label Distribution:
O                              38616
I-EMAIL                        3592
I-SKILLS                       2003
I-DESIGNATION                  711
I-COLLEGE_NAME                 451
I-NAME                         440
I-DEGREE                       391
I-COMPANY                      352
B-COMPANY                      317
B-DESIGNATION                  301
B-LOCATION                     277
B-NAME                         175
B-EMAIL                        171
I-LOCATION                     162
B-COLLEGE_NAME                 125
B-SKILLS                       123
B-DEGREE                       104
B-GRADUATION_YEAR              104
I-YEARS_OF_EXPERIENCE          68
B-YEARS_OF_EXPERIENCE          31
I-GRADUATION_YEAR              3


In [7]:
# Display a sample of tokenized input and corresponding labels
sample = train_dataset[0]

tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])

print(f"{'TOKEN':<25} LABEL")
print("-" * 50)

shown = 0

for token, label_id in zip(tokens, sample["labels"]):

    if label_id == -100:
        continue

    label = id2label[label_id]

    print(f"{token:<25} {label}")

    shown += 1

    if shown >= 80:
        break

TOKEN                     LABEL
--------------------------------------------------
▁Sharan                   B-NAME
▁Ad                       I-NAME
la                        I-NAME
▁-                        O
▁Email                    O
▁me                       O
▁on                       O
▁Indeed                   O
:                         O
▁indeed                   B-EMAIL
.                         I-EMAIL
com                       I-EMAIL
/                         I-EMAIL
r                         I-EMAIL
/                         I-EMAIL
S                         I-EMAIL
haran                     I-EMAIL
-                         I-EMAIL
Ad                        I-EMAIL
la                        I-EMAIL
/                         I-EMAIL
3                         I-EMAIL
a                         I-EMAIL
382                       I-EMAIL
a                         I-EMAIL
7                         I-EMAIL
b                         I-EMAIL
7                         I-EMAIL
296 

In [24]:
import torch
from transformers import AutoModelForTokenClassification

fresh_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id
)

print("1. Immediately after loading:")
print(next(fresh_model.parameters()).dtype)

fresh_model = fresh_model.to(dtype=torch.float32)

print("2. After forcing float32:")
print(next(fresh_model.parameters()).dtype)

fresh_model = fresh_model.to("cuda")

print("3. After moving to GPU:")
print(next(fresh_model.parameters()).dtype)
print(next(fresh_model.parameters()).device)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

1. Immediately after loading:
torch.float16
2. After forcing float32:
torch.float32
3. After moving to GPU:
torch.float32
cuda:0


In [23]:
from collections import Counter

print(Counter(
    str(p.dtype)
    for p in fresh_model.parameters()
))

Counter({'torch.float32': 200})


In [26]:
print("Model dtype:", next(model.parameters()).dtype)
print("Device:", next(model.parameters()).device)

Model dtype: torch.float16
Device: cuda:0


In [27]:
# Evaluation metrics
def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        current_predictions = []
        current_labels = []

        for p, l in zip(prediction, label):

            if l == -100:
                continue

            current_predictions.append(id2label[p])
            current_labels.append(id2label[l])

        true_predictions.append(current_predictions)
        true_labels.append(current_labels)

    return {
        "precision": precision_score(
            true_labels,
            true_predictions
        ),
        "recall": recall_score(
            true_labels,
            true_predictions
        ),
        "f1": f1_score(
            true_labels,
            true_predictions
        ),
        "accuracy": accuracy_score(
            true_labels,
            true_predictions
        )
    }

In [28]:
# Data Collator
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

print("Data collator ready.")

Data collator ready.


In [29]:
from torch.utils.data import DataLoader
import torch

test_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=data_collator
)

batch = next(iter(test_loader))

batch = {
    key: value.cuda()
    for key, value in batch.items()
}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id
).cuda()

print("Model dtype:", next(model.parameters()).dtype)

with torch.no_grad():
    outputs = model(**batch)

print("Loss:", outputs.loss.item())
print("Loss is NaN:", torch.isnan(outputs.loss).item())
print("Logits contain NaN:", torch.isnan(outputs.logits).any().item())

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

Model dtype: torch.float16
Loss: 4.2109375
Loss is NaN: False
Logits contain NaN: False


In [30]:
# Hyperparameter tuning trials
hyperparameter_trials = [
    # Learning-rate comparison
    {
        "learning_rate": 1e-5,
        "batch_size": 4,
        "epochs": 3,
        "weight_decay": 0.01
    },
    {
        "learning_rate": 2e-5,
        "batch_size": 4,
        "epochs": 3,
        "weight_decay": 0.01
    },
    {
        "learning_rate": 3e-5,
        "batch_size": 4,
        "epochs": 3,
        "weight_decay": 0.01
    },
    {
        "learning_rate": 5e-5,
        "batch_size": 4,
        "epochs": 3,
        "weight_decay": 0.01
    },

    # Epoch comparison
    {
        "learning_rate": 2e-5,
        "batch_size": 4,
        "epochs": 2,
        "weight_decay": 0.01
    },
    {
        "learning_rate": 2e-5,
        "batch_size": 4,
        "epochs": 4,
        "weight_decay": 0.01
    },

    # Weight-decay comparison
    {
        "learning_rate": 2e-5,
        "batch_size": 4,
        "epochs": 3,
        "weight_decay": 0.05
    },
    {
        "learning_rate": 3e-5,
        "batch_size": 4,
        "epochs": 4,
        "weight_decay": 0.05
    }
]

print("Number of tuning trials:", len(hyperparameter_trials))

Number of tuning trials: 8


In [ ]:
# Hyperparameter tuning loop
import time
import gc
import pandas as pd
import torch

from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

tuning_results = []

best_f1 = -1
best_trial = None

for trial_number, params in enumerate(hyperparameter_trials, start=1):

    print("\n" + "=" * 70)
    print(f"TRIAL {trial_number}")
    print("=" * 70)

    print("Learning Rate :", params["learning_rate"])
    print("Batch Size    :", params["batch_size"])
    print("Epochs        :", params["epochs"])
    print("Weight Decay  :", params["weight_decay"])

    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_LIST),
        id2label=id2label,
        label2id=label2id
    )

    # IMPORTANT: force model to FP32
    model = model.to(dtype=torch.float32)

    print("Model dtype:", next(model.parameters()).dtype)

    training_args = TrainingArguments(
        output_dir=f"../models/benchmark_a/trial_{trial_number}",

        learning_rate=params["learning_rate"],

        per_device_train_batch_size=params["batch_size"],
        per_device_eval_batch_size=params["batch_size"],

        num_train_epochs=params["epochs"],
        weight_decay=params["weight_decay"],

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,

        save_total_limit=1,
        logging_steps=20,

        fp16=False,

        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    start_time = time.time()

    trainer.train()

    training_time = time.time() - start_time

    validation_results = trainer.evaluate(val_dataset)

    trial_result = {
        "Trial": trial_number,
        "Learning Rate": params["learning_rate"],
        "Batch Size": params["batch_size"],
        "Epochs": params["epochs"],
        "Weight Decay": params["weight_decay"],
        "Validation Precision": validation_results["eval_precision"],
        "Validation Recall": validation_results["eval_recall"],
        "Validation F1": validation_results["eval_f1"],
        "Validation Accuracy": validation_results["eval_accuracy"],
        "Training Time (seconds)": training_time
    }

    tuning_results.append(trial_result)

    print("\nValidation Results")
    print("Precision:", validation_results["eval_precision"])
    print("Recall:", validation_results["eval_recall"])
    print("F1:", validation_results["eval_f1"])

    if validation_results["eval_f1"] > best_f1:

        best_f1 = validation_results["eval_f1"]
        best_trial = trial_result

        trainer.save_model(
            "../models/benchmark_a/best_model"
        )

        tokenizer.save_pretrained(
            "../models/benchmark_a/best_model"
        )

        print("\nNEW BEST MODEL SAVED")

    del trainer
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


TRIAL 1
Learning Rate : 1e-05
Batch Size    : 4
Epochs        : 3
Weight Decay  : 0.01


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task

Model dtype: torch.float32


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.176049,0.273852,4,0.493274,0.550000,0.520095,0.920745



Validation Results
Precision: 0.49327354260089684
Recall: 0.55
F1: 0.5200945626477542


In [32]:
# Show and Save tuning results
import os
import pandas as pd

os.makedirs("../results", exist_ok=True)

tuning_df = pd.DataFrame(tuning_results)

tuning_df = tuning_df.sort_values(
    by="Validation F1",
    ascending=False
).reset_index(drop=True)

display(tuning_df)

tuning_df.to_csv(
    "../results/benchmark_a_tuning.csv",
    index=False
)

print("Saved: ../results/benchmark_a_tuning.csv")

,Trial,Learning Rate,Batch Size,Epochs,Weight Decay,Validation Precision,Validation Recall,Validation F1,Validation Accuracy,Training Time (seconds)
0,4,0.00005,4,3,0.01,0.513274,0.580,0.544601,0.916342,92.968496
1,8,0.00003,4,4,0.05,0.493274,0.550,0.520095,0.920745,130.113817
2,6,0.00002,4,4,0.01,0.500000,0.525,0.512195,0.910923,126.145768
3,3,0.00003,4,3,0.01,0.435897,0.510,0.470046,0.915665,98.223093
4,2,0.00002,4,3,0.01,0.431111,0.485,0.456471,0.910415,93.296803
5,7,0.00002,4,3,0.05,0.424779,0.480,0.450704,0.908721,94.274794
6,5,0.00002,4,2,0.01,0.387387,0.430,0.407583,0.892464,62.251089
7,1,0.00001,4,3,0.01,0.484375,0.310,0.378049,0.877223,94.571164


Saved: ../results/benchmark_a_tuning.csv


In [33]:
# Show winner
print("Best Hyperparameter Configuration")
print("=" * 50)

for key, value in best_trial.items():
    print(f"{key}: {value}")

print(f"\nBest Validation F1: {best_f1:.4f}")

Best Hyperparameter Configuration
Trial: 4
Learning Rate: 5e-05
Batch Size: 4
Epochs: 3
Weight Decay: 0.01
Validation Precision: 0.5132743362831859
Validation Recall: 0.58
Validation F1: 0.5446009389671361
Validation Accuracy: 0.916342082980525
Training Time (seconds): 92.9684956073761

Best Validation F1: 0.5446


In [ ]:
# Reload the best saved DeBERTa model
import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

BEST_MODEL_PATH = "../models/benchmark_a/best_model"

best_tokenizer = AutoTokenizer.from_pretrained(
    BEST_MODEL_PATH
)

best_model = AutoModelForTokenClassification.from_pretrained(
    BEST_MODEL_PATH
)

# Keep model stable in FP32
best_model = best_model.to(dtype=torch.float32)

print("Best model loaded.")
print("Model dtype:", next(best_model.parameters()).dtype)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Best model loaded.
Model dtype: torch.float32


In [35]:
# Data Collator for the best model
final_data_collator = DataCollatorForTokenClassification(
    tokenizer=best_tokenizer
)

In [36]:
# Create final test Trainer
final_args = TrainingArguments(
    output_dir="../models/benchmark_a/final_test",
    per_device_eval_batch_size=4,
    fp16=False,
    bf16=False,
    report_to="none"
)

final_trainer = Trainer(
    model=best_model,
    args=final_args,
    eval_dataset=test_dataset,
    processing_class=best_tokenizer,
    data_collator=final_data_collator,
    compute_metrics=compute_metrics
)

print("Final test trainer ready.")

Final test trainer ready.


In [37]:
# Evaluate the best model on the test dataset
test_results = final_trainer.evaluate(test_dataset)

print("\nFINAL BENCHMARK A — DeBERTa-v3-base")
print("=" * 55)

print(f"Test Loss : {test_results['eval_loss']:.4f}")
print(f"Precision : {test_results['eval_precision']:.4f}")
print(f"Recall    : {test_results['eval_recall']:.4f}")
print(f"F1 Score  : {test_results['eval_f1']:.4f}")
print(f"Accuracy  : {test_results['eval_accuracy']:.4f}")

Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,0.268785,0,0.550459,0.612245,0.579710,0.927132



FINAL BENCHMARK A — DeBERTa-v3-base
Test Loss : 0.2688
Precision : 0.5505
Recall    : 0.6122
F1 Score  : 0.5797
Accuracy  : 0.9271


In [40]:
# Final TEST evaluation
test_results = final_trainer.evaluate(test_dataset)

print("\nBENCHMARK A — DeBERTa-v3-base")
print("=" * 50)

print(f"Test Loss : {test_results['eval_loss']:.4f}")
print(f"Precision : {test_results['eval_precision']:.4f}")
print(f"Recall    : {test_results['eval_recall']:.4f}")
print(f"F1 Score  : {test_results['eval_f1']:.4f}")
print(f"Accuracy  : {test_results['eval_accuracy']:.4f}")

Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,0.268785,0,0.550459,0.612245,0.579710,0.927132



BENCHMARK A — DeBERTa-v3-base
Test Loss : 0.2688
Precision : 0.5505
Recall    : 0.6122
F1 Score  : 0.5797
Accuracy  : 0.9271


In [43]:
# Generate classification report
import numpy as np
from seqeval.metrics import classification_report

prediction_output = final_trainer.predict(test_dataset)

predictions = np.argmax(
    prediction_output.predictions,
    axis=2
)

labels = prediction_output.label_ids

true_predictions = []
true_labels = []

for prediction, label in zip(predictions, labels):

    predicted_sequence = []
    true_sequence = []

    for predicted_id, true_id in zip(prediction, label):

        if true_id == -100:
            continue

        predicted_sequence.append(
            id2label[int(predicted_id)]
        )

        true_sequence.append(
            id2label[int(true_id)]
        )

    true_predictions.append(predicted_sequence)
    true_labels.append(true_sequence)

report_dict = classification_report(
    true_labels,
    true_predictions,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).T

display(
    report_df.round(4)
)

,precision,recall,f1-score,support
COLLEGE_NAME,0.3000,0.3000,0.3000,10.0
COMPANY,0.3898,0.5610,0.4600,41.0
DEGREE,0.6667,0.7500,0.7059,8.0
DESIGNATION,0.6222,0.5957,0.6087,47.0
EMAIL,0.7273,0.9412,0.8205,17.0
GRADUATION_YEAR,0.5000,0.0909,0.1538,11.0
LOCATION,0.5143,0.6923,0.5902,26.0
NAME,0.9545,0.9130,0.9333,23.0
SKILLS,0.3077,0.3636,0.3333,11.0
YEARS_OF_EXPERIENCE,0.0000,0.0000,0.0000,2.0


In [ ]:
# Save per-entity results
report_df.to_csv(
    "../results/benchmark_a_per_entity.csv"
)

print("Saved per-entity results.")

Saved per-entity results.


In [45]:
# Save overall metrics
benchmark_a_metrics = pd.DataFrame(
    [{
        "precision": test_results["eval_precision"],
        "recall": test_results["eval_recall"],
        "f1": test_results["eval_f1"],
        "accuracy": test_results["eval_accuracy"]
    }],
    index=["DeBERTa-v3"]
)

display(
    benchmark_a_metrics.round(4)
)

benchmark_a_metrics.to_csv(
    "../results/benchmark_a_test_metrics.csv"
)

print("Saved Benchmark A test metrics.")

,precision,recall,f1,accuracy
DeBERTa-v3,0.5505,0.6122,0.5797,0.9271


Saved Benchmark A test metrics.


In [46]:
# Save benchmark_a.json
import json

benchmark_a_results = {
    "model": "microsoft/deberta-v3-base",

    "best_hyperparameters": best_trial,
    "best_validation_f1": float(best_f1),

    "test_loss": float(test_results["eval_loss"]),
    "test_precision": float(test_results["eval_precision"]),
    "test_recall": float(test_results["eval_recall"]),
    "test_f1": float(test_results["eval_f1"]),
    "test_accuracy": float(test_results["eval_accuracy"])
}

with open(
    "../results/benchmark_a.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        benchmark_a_results,
        f,
        indent=4
    )

print("Saved benchmark_a.json")

Saved benchmark_a.json


In [47]:
# Verify before deleting the huge trial folders
print(
    "Best model:",
    os.path.exists("../models/benchmark_a/best_model")
)

print(
    "Tuning results:",
    os.path.exists("../results/benchmark_a_tuning.csv")
)

print(
    "Overall test:",
    os.path.exists("../results/benchmark_a_test_metrics.csv")
)

print(
    "Per-entity test:",
    os.path.exists("../results/benchmark_a_per_entity.csv")
)

print(
    "JSON:",
    os.path.exists("../results/benchmark_a.json")
)

Best model: True
Tuning results: True
Overall test: True
Per-entity test: True
JSON: True
